# Lab 0B Assignment: Revise a Prompt for More Consistent Results

Use this notebook after you finish `02_model_comparison.ipynb`.

What stays the same from `02_model_comparison.ipynb`:
- the same three class models
- the same synthetic case note
- the same baseline prompt structure
- the same basic workflow of running the models and reviewing raw outputs

What is new in this notebook:
- you revise the prompt yourself
- you rerun the same three models with the revised prompt
- you compare the before/after results with a simple summary

In this assignment, you will:
- use the same three class models from `02_model_comparison.ipynb`
- run the original baseline prompt on those three models
- identify one inconsistency in the results
- revise the prompt to make the outputs more consistent
- rerun the same three models
- compare the before/after results with a simple summary

In [ ]:
import json
from pathlib import Path
from time import perf_counter

import requests
from dotenv import dotenv_values
from openai import OpenAI

# Teaching note:
# This setup cell finds the repo root, loads .env, and builds the model client.
# The same settings are reused for both the baseline and revised prompt runs.
cwd = Path.cwd().resolve()
repo_root = cwd.parent

if not (repo_root / '.env.example').exists() or not (repo_root / 'src').exists():
    raise FileNotFoundError(
        'Open this notebook from its lab folder so the repo root is one level up.'
    )

config = dotenv_values(repo_root / '.env')
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError('MODEL or OLLAMA_BASE_URL is missing from .env')

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Default model from .env:', default_model)
print('Ollama endpoint:', ollama_base_url)


## Step 1: Discover Available Models

Run the next cell to list the models available from the configured Ollama endpoint.

In [ ]:
# Teaching note:
# This block asks Ollama for the available model list used in the warm-up.
# It turns the raw server response into a simple list the notebook can validate.
tags_url = ollama_base_url.rstrip('/').replace('/v1', '/api/tags')

response = requests.get(tags_url, timeout=10)
response.raise_for_status()
available_models = [item.get('name') for item in response.json().get('models', []) if item.get('name')]

print('Available models:')
for index, model_name in enumerate(available_models, start=1):
    print(f'{index}. {model_name}')

if len(available_models) < 3:
    raise ValueError('This assignment needs at least 3 available models.')


## Step 2: Choose Three Models

Use the same three class models from `02_model_comparison.ipynb` so everyone in the class works with the same set:
- `qwen3.5:0.8b`
- `qwen3.5:27b`
- `gemma4:e4b`

The next cell checks whether all three are available at the configured endpoint.

In [ ]:
# Teaching note:
# This block locks the notebook to the fixed class model set.
# It also checks whether any required model is missing from the endpoint.
models_to_compare = ['qwen3.5:0.8b', 'qwen3.5:27b', 'gemma4:e4b']

missing_models = [name for name in models_to_compare if name not in available_models]

print('Models selected for comparison:')
for model_name in models_to_compare:
    print('-', model_name)

if missing_models:
    raise ValueError(f'These required models are missing from the endpoint: {missing_models}')

if len(set(models_to_compare)) != 3:
    raise ValueError('Choose 3 different models before continuing.')


## Step 3: Synthetic Case Note

Use the same synthetic note for both the baseline prompt and your revised prompt.

In [ ]:
case_note = """
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.
""".strip()

print(case_note)

## Step 4: Baseline Prompt

Run the original prompt first so you have a baseline for comparison.

In [ ]:
baseline_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

print(baseline_prompt)

## Step 5: Run the Baseline Prompt

This cell sends the original prompt to each of your three models and prints their raw outputs.

In [ ]:
# Teaching note:
# clean_json_text(...) trims extra Markdown or wrapper text around model JSON.
# That makes the responses easier to parse and compare before and after prompt revision.
def clean_json_text(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned

# Teaching note:
# ask_model(...) sends one prompt to one model and records the result.
# It keeps timing, raw text, and parse status together for before and after comparison.
def ask_model(model_name: str, prompt: str) -> dict:
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
    )
    elapsed = perf_counter() - start
    raw_text = response.choices[0].message.content
    cleaned_text = clean_json_text(raw_text)

    try:
        parsed = json.loads(cleaned_text)
        parse_error = None
    except Exception as exc:
        parsed = None
        parse_error = str(exc)

    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
        'parsed': parsed,
        'parse_error': parse_error,
    }

# Teaching note:
# run_prompt_for_models(...) applies the same prompt to all selected models.
# This keeps the baseline and revised runs comparable.
def run_prompt_for_models(prompt_name: str, prompt: str) -> list[dict]:
    results = []
    for model_name in models_to_compare:
        print(f'Running {prompt_name} prompt on {model_name}...')
        result = ask_model(model_name, prompt)
        result['prompt_name'] = prompt_name
        results.append(result)
    return results

baseline_results = run_prompt_for_models('baseline', baseline_prompt)

print('\nBaseline raw outputs:')
for result in baseline_results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()


## Step 6: Revise the Prompt

Edit the prompt below before running the next cell.

Goal: make the results more consistent across models.

A good place to focus is `device_ids`, because the baseline prompt does not clearly say whether labels such as `IMEI` or `serial number` should be included.

In [ ]:
# Teaching note:
# Edit this prompt before Step 7.
# Add at least one rule that makes the device ID output more consistent across models.
revised_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

if revised_prompt == baseline_prompt:
    print('Reminder: revise the prompt before running Step 7.')

print(revised_prompt)


## Step 7: Run the Revised Prompt and Compare the Results

This cell reruns the same three models with your revised prompt and builds a simple before/after comparison summary.

For each model, the summary shows:
- response time in seconds
- whether the response could be parsed as JSON
- which keys were returned
- what the model put in `device_ids`

In [ ]:
revised_results = run_prompt_for_models('revised', revised_prompt)

print('\nRevised raw outputs:')
for result in revised_results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()

baseline_summary = []
for result in baseline_results:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []
    baseline_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'device_ids_found': device_ids_found,
    })

revised_summary = []
for result in revised_results:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []
    revised_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'device_ids_found': device_ids_found,
    })

comparison_by_model = []
for model_name in models_to_compare:
    baseline_item = next(item for item in baseline_summary if item['model'] == model_name)
    revised_item = next(item for item in revised_summary if item['model'] == model_name)
    comparison_by_model.append({
        'model': model_name,
        'baseline': baseline_item,
        'revised': revised_item,
    })

print('\nBefore/after comparison summary:')
print(json.dumps(comparison_by_model, indent=2))

## Reflection Questions

Replace this text with short answers to the questions below.

Use the before/after summary and the raw outputs to support your answers.

1. What inconsistency did you notice in the baseline outputs?
2. What specific rule did you add to the revised prompt?
3. Did the revised prompt make the `device_ids` format more consistent? Give one example.
4. Did JSON validity or returned keys change from baseline to revised?
5. Which model changed the most after your prompt revision, and what changed?

## Submission

Save the notebook with:
- the three selected models
- the baseline prompt and baseline raw outputs
- your revised prompt
- the revised raw outputs
- the summary comparison
- your short reflection